In [1]:

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp


In [2]:
CSV_PATH = "output.csv"     # path to your noise-free figure-eight run
COL_T = "time"              # time column name
COL_UR = "u_R"              # right wheel input column name
COL_UL = "u_L"              # left wheel input column name
COL_WR = "omega_R"          # right wheel speed column name
COL_WL = "omega_L"

In [3]:
# Nominal parameter values: [kgR, kgL, ka, Kf, Kq]

ETA_NOMINAL = np.array([25.0, 25.0, 0.5, 0.15, 0.02])
ALPHA = 8.0                          # tanh steepness constant (beta = alpha, shared)
 
PARAM_NAMES = ["kgR", "kgL", "ka", "Kf", "Kq"]

In [4]:
df = pd.read_csv(CSV_PATH, skiprows=10)
t = df[COL_T].to_numpy()
U_R = df[COL_UR].to_numpy()
U_L = df[COL_UL].to_numpy()
w_R = df[COL_WR].to_numpy()
w_L = df[COL_WL].to_numpy()
N = len(t)

In [5]:
def sech2(x):
    return 1.0 / np.cosh(x) ** 2
 
def A_term(w, ka, Kf, Kq, alpha):
    """A(t) = d(dw/dt)/dw -- shared 'memory/decay' coefficient."""
    return (
        -ka
        - Kf * alpha * sech2(alpha * w)
        - Kq * (alpha * w**2 * sech2(alpha * w) + 2 * w * np.tanh(alpha * w))
    )
 
def b_vec_R(w, U, alpha):
    """d(dwR/dt)/d[kgR,kgL,ka,Kf,Kq] -- instantaneous push, right wheel."""
    return np.array([U, 0.0, -w, -np.tanh(alpha * w), -np.tanh(alpha * w) * w**2])
 
def b_vec_L(w, U, alpha):
    """d(dwL/dt)/d[kgR,kgL,ka,Kf,Kq] -- instantaneous push, left wheel."""
    return np.array([0.0, U, -w, -np.tanh(alpha * w), -np.tanh(alpha * w) * w**2])

In [6]:
ka, Kf, Kq = ETA_NOMINAL[2], ETA_NOMINAL[3], ETA_NOMINAL[4]
 
S = np.zeros((N, 2, 5))   # S[i] = 2x5 sensitivity matrix at timestep i
S[0] = 0.0                # initial condition: no sensitivity at t=0
 
for i in range(1, N):
    dt = t[i] - t[i - 1]
 
    # Use previous timestep's state to step forward (explicit Euler)
    A_R = A_term(w_R[i - 1], ka, Kf, Kq, ALPHA)
    A_L = A_term(w_L[i - 1], ka, Kf, Kq, ALPHA)
    b_R = b_vec_R(w_R[i - 1], U_R[i - 1], ALPHA)
    b_L = b_vec_L(w_L[i - 1], U_L[i - 1], ALPHA)
 
    S_R_prev = S[i - 1, 0, :]
    S_L_prev = S[i - 1, 1, :]
 
    S[i, 0, :] = S_R_prev + dt * (A_R * S_R_prev + b_R)
    S[i, 1, :] = S_L_prev + dt * (A_L * S_L_prev + b_L)
 
print(f"Sensitivity accumulation done. S has shape {S.shape} (timesteps, 2 wheels, 5 params).")
 
# Optionally save S(t) to CSV for inspection
S_flat = S.reshape(N, 10)
S_cols = [f"S_{ch}_{p}" for ch in ["wR", "wL"] for p in PARAM_NAMES]
S_df = pd.DataFrame(S_flat, columns=S_cols)
S_df.insert(0, "t", t)
S_df.to_csv("sensitivity_over_time.csv", index=False)
print("Saved sensitivity_over_time.csv")

Sensitivity accumulation done. S has shape (20000, 2, 5) (timesteps, 2 wheels, 5 params).
Saved sensitivity_over_time.csv


In [7]:
Sigma_inv = np.eye(2)   # <-- edit if wR, wL have different noise levels
 
F = np.zeros((5, 5))
for i in range(N):
    if i == 0:
        continue
    dt = t[i] - t[i - 1]
    Si = S[i]                          # 2x5
    F += Si.T @ Sigma_inv @ Si * dt    # 5x5 contribution
 
print("\nFisher Information Matrix F:")
print(F)


Fisher Information Matrix F:
[[ 3.69716797e-01  0.00000000e+00 -1.42516941e+01 -1.08389621e+01
  -1.88470598e+01]
 [ 0.00000000e+00  3.69716797e-01 -1.42516941e+01 -1.08389621e+01
  -1.88470598e+01]
 [-1.42516941e+01 -1.42516941e+01  1.10396672e+03  8.35882446e+02
   1.46309475e+03]
 [-1.08389621e+01 -1.08389621e+01  8.35882446e+02  6.35550930e+02
   1.10552359e+03]
 [-1.88470598e+01 -1.88470598e+01  1.46309475e+03  1.10552359e+03
   1.94117250e+03]]


In [8]:
eigvals, eigvecs = np.linalg.eigh(F)   # ascending order
print("\nEigenvalues (smallest = most unidentifiable direction):")
for val in eigvals:
    print(f"  {val:.6e}")
 
print("\nEigenvectors (columns), paired with eigenvalues above:")
print("Params order:", PARAM_NAMES)
for j in range(5):
    print(f"\nEigenvalue {eigvals[j]:.4e}, eigenvector:")
    for name, comp in zip(PARAM_NAMES, eigvecs[:, j]):
        print(f"  {name}: {comp:+.4f}")


Eigenvalues (smallest = most unidentifiable direction):
  2.716494e-06
  2.937374e-02
  3.697168e-01
  4.543432e+00
  3.676487e+03

Eigenvectors (columns), paired with eigenvalues above:
Params order: ['kgR', 'kgL', 'ka', 'Kf', 'Kq']

Eigenvalue 2.7165e-06, eigenvector:
  kgR: +0.7068
  kgL: +0.7068
  ka: -0.0096
  Kf: +0.0293
  Kq: +0.0043

Eigenvalue 2.9374e-02, eigenvector:
  kgR: -0.0145
  kgL: -0.0145
  ka: -0.8310
  Kf: +0.3681
  Kq: +0.4164

Eigenvalue 3.6972e-01, eigenvector:
  kgR: +0.7071
  kgL: -0.7071
  ka: +0.0000
  Kf: -0.0000
  Kq: -0.0000

Eigenvalue 4.5434e+00, eigenvector:
  kgR: +0.0149
  kgL: +0.0149
  ka: -0.0949
  Kf: -0.8316
  Kq: +0.5467

Eigenvalue 3.6765e+03, eigenvector:
  kgR: -0.0071
  kgL: -0.0071
  ka: +0.5480
  Kf: +0.4147
  Kq: +0.7264


In [9]:
# STEP 6 (VALIDATION): Finite-difference check against analytic S(t)
# Re-integrates the base ODE (not the sensitivity ODE) at nominal and
# bumped parameter values, using U_R(t), U_L(t) interpolated from your CSV.
# ============================================================
U_R_func = interp1d(t, U_R, kind="linear", fill_value="extrapolate")
U_L_func = interp1d(t, U_L, kind="linear", fill_value="extrapolate")
 
def base_ode(t_, y, eta, alpha):
    wR, wL = y
    kgR, kgL, ka_, Kf_, Kq_ = eta
    UR = U_R_func(t_)
    UL = U_L_func(t_)
    dwR = kgR * UR - ka_ * wR - Kf_ * np.tanh(alpha * wR) - Kq_ * np.tanh(alpha * wR) * wR**2
    dwL = kgL * UL - ka_ * wL - Kf_ * np.tanh(alpha * wL) - Kq_ * np.tanh(alpha * wL) * wL**2
    return [dwR, dwL]
 
def run_sim(eta, alpha, t_eval):
    y0 = [w_R[0], w_L[0]]   # match your CSV's initial condition
    sol = solve_ivp(base_ode, [t_eval[0], t_eval[-1]], y0, args=(eta, alpha),
                     t_eval=t_eval, method="RK45", rtol=1e-8, atol=1e-10)
    return sol.y   # shape (2, N)
 
def finite_diff_check(param_index, eps_frac=1e-3):
    eta_plus = ETA_NOMINAL.copy()
    bump = eta_plus[param_index] * eps_frac
    eta_plus[param_index] += bump
 
    traj_nominal = run_sim(ETA_NOMINAL, ALPHA, t)
    traj_bumped = run_sim(eta_plus, ALPHA, t)
 
    fd_S_wR = (traj_bumped[0] - traj_nominal[0]) / bump
    fd_S_wL = (traj_bumped[1] - traj_nominal[1]) / bump
 
    analytic_S_wR = S[:, 0, param_index]
    analytic_S_wL = S[:, 1, param_index]
 
    rel_err_wR = np.abs(fd_S_wR - analytic_S_wR) / (np.abs(fd_S_wR).max() + 1e-12)
    rel_err_wL = np.abs(fd_S_wL - analytic_S_wL) / (np.abs(fd_S_wL).max() + 1e-12)
 
    print(f"\nParam {PARAM_NAMES[param_index]}: "
          f"max rel err wR = {rel_err_wR.max():.4e}, "
          f"max rel err wL = {rel_err_wL.max():.4e}")
 
print("\n=== Finite-difference validation (should be small, e.g. <1e-2) ===")
for idx in range(5):
    finite_diff_check(idx)
 


=== Finite-difference validation (should be small, e.g. <1e-2) ===

Param kgR: max rel err wR = 5.1903e-04, max rel err wL = 9.9990e-01

Param kgL: max rel err wR = 9.9990e-01, max rel err wL = 5.1903e-04

Param ka: max rel err wR = 9.7883e-04, max rel err wL = 9.7883e-04

Param Kf: max rel err wR = 7.6834e-04, max rel err wL = 7.6834e-04

Param Kq: max rel err wR = 6.4025e-04, max rel err wL = 6.4025e-04
